# 00 — Orientation and system map

## Learning objectives

After this notebook you should be able to:

- explain what Phases A, B, and C own and why the boundaries matter;
- distinguish raw evidence, canonical publications, research results, portfolio ledgers, caches, source, and prototypes;
- identify the implementation commit and immutable data/run identities needed for reproduction;
- separate implemented and validated behavior from caveated, deferred, and absent behavior.

**Evidence examined.** Repository HEAD `00e35d98a49492a7913a1e862117c5ae19757d06`; Phase A `phasea-2a2b3898aba37814`; extended diagnostic Phase A `phasea-9a50dcdb3a4538d7`; GPW Phase B `phaseb-f88fc2d38e9811ed1573`; U.S. Phase B `phaseb-5d7086751156ac48cef3`; Phase C `phasec-fa439d650410376aae9e` and its `00e35d9` reproduction.

This is an evidence-guided owner walkthrough, not generated API documentation. It performs no publication writes.

## Configuration

The next cell is the only path configuration. Environment variables with the displayed names override the established Windows defaults. It also makes `source/python/src` importable without installing or changing the environment.

In [1]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd

REPO_ROOT = Path(os.environ.get("ATS_REPO_ROOT", r"D:\Stock\ATS"))
DATA_ROOT = Path(os.environ.get("ATS_DATA_ROOT", r"D:\Stock\data\ATS"))
PROJECT_ROOT = REPO_ROOT / "source" / "python"
RESEARCH_ROOT = REPO_ROOT / "RESEARCH"
GPW_MANIFEST = Path(os.environ.get(
    "ATS_GPW_MANIFEST",
    DATA_ROOT / "phase_b" / "versions" / "phaseb-f88fc2d38e9811ed1573" / "manifest.json",
))
US_MANIFEST = Path(os.environ.get(
    "ATS_US_MANIFEST",
    DATA_ROOT / "phase_b" / "versions" / "phaseb-5d7086751156ac48cef3" / "manifest.json",
))
PHASE_A_RUN = Path(os.environ.get(
    "ATS_PHASE_A_RUN", DATA_ROOT / "phase_a" / "runs" / "phasea-2a2b3898aba37814"
))
PHASE_A_EXTENDED_RUN = Path(os.environ.get(
    "ATS_PHASE_A_EXTENDED_RUN",
    DATA_ROOT / "decision_oriented_phase_a" / "runs" / "extension-20260820T163347Z",
))
PHASE_C_RUN = Path(os.environ.get(
    "ATS_PHASE_C_RUN", DATA_ROOT / "phase_c" / "runs" / "phasec-fa439d650410376aae9e"
))
PHASE_C_REPRODUCTION = Path(os.environ.get(
    "ATS_PHASE_C_REPRODUCTION",
    DATA_ROOT / "phase_c" / "reproductions" / "00e35d9" / "phasec-fa439d650410376aae9e",
))

src = str(PROJECT_ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)

native_library_dir = str(Path(sys.prefix) / "Library" / "bin")
path_entries = [entry.rstrip("\\").lower() for entry in os.environ.get("PATH", "").split(os.pathsep)]
if native_library_dir.rstrip("\\").lower() not in path_entries:
    raise RuntimeError(f"Conda native-library directory is absent from kernel PATH: {native_library_dir}")

required = [REPO_ROOT, DATA_ROOT, GPW_MANIFEST, PHASE_A_RUN, PHASE_C_RUN]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Required retained evidence is missing: {missing}")

pd.set_option("display.max_rows", 12)
pd.set_option("display.max_columns", 14)
pd.set_option("display.width", 140)
print({
    "repo": str(REPO_ROOT),
    "data": str(DATA_ROOT),
    "python": sys.version.split()[0],
    "native_library_path": True,
    "jupyter_runtime": os.environ.get("JUPYTER_RUNTIME_DIR"),
})

{'repo': 'D:\\Stock\\ATS', 'data': 'D:\\Stock\\data\\ATS', 'python': '3.12.13', 'native_library_path': True, 'jupyter_runtime': 'D:\\Stock\\ATS\\RESEARCH\\.tmp\\ats-env\\jupyter'}


### Fresh-kernel native-library smoke test

This deliberately exercises NumPy linear algebra and SciPy statistics. Both reach delayed native DLLs that failed to load when the unrepaired kernelspec started Python without the Conda `Library\bin` directory.

In [2]:
import numpy as np
from scipy import stats

singular_values = np.linalg.svd(np.eye(3), compute_uv=False)
rank_result = stats.spearmanr([1, 2, 3], [3, 2, 1])
print({
    "svd": singular_values.tolist(),
    "spearman": float(rank_result.statistic),
    "native_dll_smoke": "PASS",
})
assert singular_values.tolist() == [1.0, 1.0, 1.0]
assert float(rank_result.statistic) == -1.0

{'svd': [1.0, 1.0, 1.0], 'spearman': -1.0, 'native_dll_smoke': 'PASS'}


## Current implementation and publication identities

Git provenance and data provenance answer different questions: Git says *which implementation*; manifests and run IDs say *which immutable inputs and logical outputs*. Reproducible research needs both.

In [3]:
head = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, check=True, text=True, capture_output=True
).stdout.strip()
baseline_head = "00e35d98a49492a7913a1e862117c5ae19757d06"
baseline_is_ancestor = subprocess.run(
    ["git", "merge-base", "--is-ancestor", baseline_head, head], cwd=REPO_ROOT
).returncode == 0
status = subprocess.run(
    ["git", "status", "--short"], cwd=REPO_ROOT, check=True, text=True, capture_output=True
).stdout.splitlines()

phase_a = json.loads((PHASE_A_RUN / "manifest.json").read_text(encoding="utf-8"))
phase_a_ext = json.loads((PHASE_A_EXTENDED_RUN / "manifest.json").read_text(encoding="utf-8"))
gpw = json.loads(GPW_MANIFEST.read_text(encoding="utf-8"))
us = json.loads(US_MANIFEST.read_text(encoding="utf-8"))
phase_c = json.loads((PHASE_C_RUN / "manifest.json").read_text(encoding="utf-8"))

identities = pd.DataFrame([
    {"evidence": "checkout", "identity": head, "implementation_commit": head, "clean_publication": None},
    {"evidence": "Phase A", "identity": phase_a["run_id"], "implementation_commit": phase_a["git_commit"], "clean_publication": not phase_a["git_state"]["dirty"]},
    {"evidence": "Phase A extended", "identity": phase_a_ext["run_id"], "implementation_commit": phase_a_ext["git_commit"], "clean_publication": not phase_a_ext["git_state"]["dirty"]},
    {"evidence": "Phase B GPW", "identity": gpw["dataset_version_id"], "implementation_commit": gpw["implementation_provenance"]["commit"], "clean_publication": gpw["implementation_provenance"]["clean"]},
    {"evidence": "Phase B U.S.", "identity": us["dataset_version_id"], "implementation_commit": us["implementation_provenance"]["commit"], "clean_publication": us["implementation_provenance"]["clean"]},
    {"evidence": "Phase C", "identity": phase_c["run_id"], "implementation_commit": phase_c["implementation_provenance"]["commit"], "clean_publication": phase_c["implementation_provenance"]["clean"]},
])
display(identities)
print("Working-tree changes intentionally visible to this review:", status)
assert baseline_is_ancestor, f"Expected Phase C review baseline {baseline_head} in current history"

,evidence,identity,implementation_commit,clean_publication
0,checkout,ea6c8d6529e351d9f2fb7c0b8b7cbb6a6425794d,ea6c8d6529e351d9f2fb7c0b8b7cbb6a6425794d,None
1,Phase A,phasea-2a2b3898aba37814,caf76ee9b7da77829cdc1b32c982a7b895e2c743,False
2,Phase A extended,phasea-9a50dcdb3a4538d7,bb82e256ad35b695b1419d3a84685da0622b9fe2,False
3,Phase B GPW,phaseb-f88fc2d38e9811ed1573,94a0e6e0792937a4b6ec8dc69c66fca85908a877,True
4,Phase B U.S.,phaseb-5d7086751156ac48cef3,94a0e6e0792937a4b6ec8dc69c66fca85908a877,True
5,Phase C,phasec-fa439d650410376aae9e,00e35d98a49492a7913a1e862117c5ae19757d06,True


Working-tree changes intentionally visible to this review: ['A  RESEARCH/ATS_REVIEW_NOTES.md', ' M source/python/README.md', 'AM source/python/notebooks/00_orientation_and_system_map.ipynb', 'AM source/python/notebooks/01_data_identity_and_point_in_time.ipynb', 'AM source/python/notebooks/02_research_findings_and_diagnostics.ipynb', 'AM source/python/notebooks/03_portfolio_ledger_and_end_to_end_flow.ipynb', 'A  source/python/notebooks/README.md', 'A  source/python/notebooks/execute_notebooks.py', 'A  source/python/notebooks/execution_report.json', '?? RESEARCH/environments/', '?? RESEARCH/prototypes/environment_repair/repaired_environment.json', '?? RESEARCH/prototypes/environment_repair/repaired_environment.yml']


**Interpretation.** The documentation-complete Phase C commit is the pinned review baseline and remains an ancestor of the current checkout; later commits retain research evidence and this walkthrough. The dirty working tree is not hidden: it contains the owner's pre-existing `source/python/README.md` edit and review-time work. Publication cleanliness is evaluated from each retained manifest's scoped provenance, not from today's unrelated files. The extended Phase A run records a different commit because it is a later retained diagnostic extension, not a rewrite of the original Phase A run.

## Compact system map

```mermaid
flowchart LR
  R[Immutable raw inputs] --> A[Phase A: research slice]
  R --> B[Phase B: canonical contracts and publications]
  A -->|exact reconciliation| B
  B --> F[Point-in-time research features and labels]
  F --> D[External portfolio decision]
  D --> C[Phase C: deterministic next-open ledger]
  C --> L[Ledgers, validation, reconciliation]
  A --> RR[Retained research reports]
  B --> M[Manifest-pinned Parquet]
```

Plain-text fallback:

```text
raw evidence -> Phase A research evidence -----> retained reports
      \-----> Phase B canonical publication -> point-in-time research
                                            -> external decision
                                            -> Phase C simulation
                                            -> ledgers + validation
```

The split is deliberate. Phase A asks research-diagnostic questions. Phase B hardens identity, time, schema, lineage, and immutable publication. Phase C consumes frozen decisions and answers accounting/execution questions. None of them silently supplies the strategy orchestration between research and an external target-weight decision.

## What the maintained areas mean

| Area | Present responsibility | Status boundary |
|---|---|---|
| `ats_contracts` | Exact Arrow and portfolio contracts; fail-closed validation | Implemented and tested |
| `ats_data` | Immutable Phase B publication, discovery, validation, GPW reconciliation, U.S. ingestion | Implemented and validated for pinned publications |
| `ats_research` | Phase A identity, universe, features, labels, diagnostics, artifacts, archive validation | Implemented for the retained GPW slice |
| `ats_portfolio` | Deterministic daily next-open execution/accounting ledger and independent validation | Implemented for frozen external intents |
| `configs` | Reference Phase A/B/C inputs with explicit identities and policies | Maintained configuration evidence |
| `tests` and fixtures | Contract, timing, denominator, publication, accounting, and state-transition guards | Passing tests prove the named conditions, not live-market fitness |
| `RESEARCH` | Architecture decisions, benchmarks, retained prototypes, Phase A findings, review notes | Durable evidence after retention audit |
| versions/runs/manifests | Immutable logical identities plus physical and logical hashes | Canonical reproducibility anchors |

There are no `ats_features` or `ats_tracking` packages. Feature responsibilities currently live in `ats_research.features`, with selected later diagnostics retained as research prototypes. Experiment tracking, packaged-engine integration, ML orchestration, and live/intraday execution remain deferred architecture.

In [4]:
areas = {
    "ats_contracts": PROJECT_ROOT / "src" / "ats_contracts",
    "ats_data": PROJECT_ROOT / "src" / "ats_data",
    "ats_research": PROJECT_ROOT / "src" / "ats_research",
    "ats_portfolio": PROJECT_ROOT / "src" / "ats_portfolio",
    "ats_features": PROJECT_ROOT / "src" / "ats_features",
    "ats_tracking": PROJECT_ROOT / "src" / "ats_tracking",
}
display(pd.DataFrame([{"module": name, "exists": path.is_dir(), "path": str(path)} for name, path in areas.items()]))

,module,exists,path
0,ats_contracts,True,D:\Stock\ATS\source\python\src\ats_contracts
1,ats_data,True,D:\Stock\ATS\source\python\src\ats_data
2,ats_research,True,D:\Stock\ATS\source\python\src\ats_research
3,ats_portfolio,True,D:\Stock\ATS\source\python\src\ats_portfolio
4,ats_features,False,D:\Stock\ATS\source\python\src\ats_features
5,ats_tracking,False,D:\Stock\ATS\source\python\src\ats_tracking


**Interpretation.** Names in an architecture report are not proof of packages. The filesystem check makes the implemented/absent distinction executable.

## Artifact classes and mutability

| Class | Example | Policy |
|---|---|---|
| Immutable raw input | local Stooq/official snapshot evidence | Read-only evidence; not silently corrected |
| Canonical publication | `D:\Stock\data\ATS\phase_b\versions\<id>` | Immutable, manifest-listed, validated |
| Research output | Phase A run and decision-oriented analysis run | Retained diagnostic evidence; not strategy returns |
| Portfolio output | Phase C run ledgers | Immutable simulation/accounting evidence |
| Disposable cache | `phase_b/cache`, `RESEARCH/.tmp`, prototype cache | Rebuildable and excluded from Git |
| Production code | `source/python/src` | Versioned implementation; unchanged by this review |
| Retained prototype | selected `RESEARCH/prototypes` source/config/tests | Evidence needed to reproduce a report, not production runtime |

## High-value tests as evidence

| Test/evidence | Failure it catches | Passing proves | Passing does not prove |
|---|---|---|---|
| `test_phase_a_archive_integrity.py` | Current checkout drift or source-snapshot tampering being mistaken for archive failure | Archived source and artifact integrity are checkable | Findings generalize beyond retained data |
| Phase A feature/label tests | Off-by-one lags, end-session leakage, missing endpoint filling | Frozen formulas and exact-session null behavior | Executability at same close |
| `test_phase_b_publication.py` | Mutable overwrite, pointer movement on failed stage, unpinned reads | Transactional immutability and manifest-driven access | Every upstream fact is correct |
| `test_phase_b_reference.py` | Lost denominator/unresolved states or provisional U.S. facts hidden | Reference reconciliation and visibility rules | U.S. issuer identity is resolved |
| Phase C golden/state tests | Same-close fill, cash/quantity mismatch, stale-value invention, action double counting | Deterministic policy behavior on named cases | Broker realism or investment merit |

Current review execution: 80/80 tests passed from the repaired environment. Later notebooks inspect the concrete cases instead of treating a green suite as a universal guarantee.

## Checkpoint classification

### Safe to rely on now

- Immutable manifest/run identity, archive integrity, and exact GPW Phase A→B reconciliation.
- Exact contracts and bounded pinned readers.
- Phase C deterministic ledger arithmetic for frozen external target weights, with independent validation and reconciliation.

### Usable with documented caveats

- GPW research diagnostics: point-in-time disciplined, but coverage/calendar-confounded and based on vendor adjustment semantics that remain unverified.
- U.S. canonical facts: bounded analytical access is validated, while identities remain provisional and issuer mapping/corporate-action metadata unresolved.
- Phase C corporate/security event policy: tested synthetically; the accepted real integration contains no corporate-action applications.

### Not implemented or not safe to rely on

- A deployable alpha claim, strategy selector, or automatic research-to-portfolio orchestrator.
- `ats_features`, `ats_tracking`, an ML pipeline, packaged-engine integration, intraday/live execution, broker integration, or Phase D.
- Authoritative U.S. issuer continuity/corporate actions or independently verified Stooq adjustment semantics.